# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 38.8691


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 439.55 GB
MemAvailable: 913.10 GB
Free GPU Memory (GB): 38.8691

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face

## 2. Generating Typos

In [3]:
exp_id = "09-01-1"

In [10]:
def process_file(file_path):
    # Read the file and count the number of rows
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    print(f"Number of rows: {len(lines)}")
    
    processed_lines = []

    # Process each line
    for line_idx, line in enumerate(lines):
        if line_idx > 10:
            break
        # Remove the trailing backslash and newline characters
        line = line.strip().rstrip('\\')
        
        # Split the line into the incorrect word and the correct words
        if '->' in line:
            incorrect_word, correct_words = line.split('->')
            
            # Split the correct words by comma and strip whitespace
            correct_words_list = [word.strip() for word in correct_words.split(',')]
            
            # Generate the output lines
            for correct_word in correct_words_list:
                processed_lines.append(f"{correct_word} {incorrect_word}")
        
    
    # Print each processed line
    for processed_line in processed_lines:
        print(processed_line)

# Define the path to your text file
file_path = "/nfs/homedirs/daro/git/quantization-reliability/data/cmw.txt"

# Run the function
process_file(file_path)

FileNotFoundError: [Errno 2] No such file or directory: '/nfs/homedirs/daro/git/quantization-reliability/data/cmw.txt'

In [6]:
import random
import string

def apply_typo_modifications(query, typo_dict):
    """
    Applies various typo modifications to a given query based on the specifications in typo_dict.
    
    Parameters:
    - query (str): The input query string to modify.
    - typo_dict (dict): A dictionary specifying the number of each type of modification to apply.
    
    Returns:
    - str: The modified query with typos introduced.
    """

    def random_insertion(word, num_insertions):
        """Randomly inserts characters into a word."""
        for _ in range(num_insertions):
            if len(word) > 1:
                pos = random.randint(0, len(word))
                char_to_insert = random.choice(string.ascii_letters)
                word = None
                if pos == 0:
                    word = char_to_insert + word
                elif pos == len(word):
                    word = word + char_to_insert
                else:
                    word = word[:pos] + char_to_insert + word[pos:]
        return word

    def random_deletion(word, num_deletions):
        """Randomly deletes characters from a word."""
        for _ in range(num_deletions):
            if len(word) > 2:  # Ensure there are at least 2 characters to avoid deleting entire word
                pos = random.randint(0, len(word) - 1)
                word = None
                if pos == 0:
                    word = word[1:]
                elif pos == len(word) - 1:
                    word = word[:-1]
                else:
                    word = word[:pos] + word[pos+1:]
        return word

    def random_replacement(word, num_replacements):
        """Randomly replaces characters in a word with adjacent keys on the keyboard."""
        keyboard_adjacency = {
            'a': 'qwsz', 'b': 'vghn', 'c': 'xdfv', 'd': 'ersfcx', 'e': 'rdsw',
            'f': 'rtgvc', 'g': 'tyhbvf', 'h': 'yujnbg', 'i': 'ujko', 'j': 'uikmnh',
            'k': 'iolmj', 'l': 'opk', 'm': 'njk', 'n': 'bhjm', 'o': 'iklp',
            'p': 'ol', 'q': 'wa', 'r': 'edft', 's': 'wedxz', 't': 'rfgy', 
            'u': 'yhji', 'v': 'cfgb', 'w': 'qase', 'x': 'zsdc', 'y': 'tghu',
            'z': 'asx', '1': '2q', '2': '13w', '3': '24e', '4': '35r', '5': '46t',
            '6': '57y', '7': '68u', '8': '79i', '9': '80o', '0': '9p'
        }
        for _ in range(num_replacements):
            if len(word) > 1:
                pos = random.randint(0, len(word) - 1)
                if word[pos].lower() in keyboard_adjacency:
                    replacement_char = random.choice(keyboard_adjacency[word[pos].lower()])
                    word = None
                    if pos == 0:
                        word = replacement_char + word[1:]
                    elif pos == len(word) - 1:
                        word = word[:-1] + replacement_char
                    else:
                        word = word[:pos] + replacement_char + word[pos+1:]
        return word

    def random_repetition(word, num_repetitions):
        """Randomly repeats characters in a word."""
        for _ in range(num_repetitions):
            if len(word) > 1:
                pos = random.randint(0, len(word))
                word = None
                if pos == 0:
                    word = word[0] + word[0] + word[1:]
                elif pos == len(word):
                    word = word + word[-1]
                else:
                    word = word[:pos] + word[pos] + word[pos:]
        return word

    def random_swapping(word, num_swaps):
        """Randomly swaps adjacent characters in a word."""
        for _ in range(num_swaps):
            if len(word) > 2:
                pos = random.randint(0, len(word) - 2)
                word = word[:pos] + word[pos+1] + word[pos] + word[pos+2:]
        return word

    def apply_cmw(word_list, num_replacements, cmw_dict):
        """Replace words in the list with their common misspelled variants if available."""
        misspellable_words = [word for word in word_list if word.lower() in cmw_dict]
        num_replacements = min(num_replacements, len(misspellable_words))  # Limit to available words

        for _ in range(num_replacements):
            word_to_misspell = random.choice(misspellable_words)
            index = word_list.index(word_to_misspell)
            word_list[index] = cmw_dict[word_to_misspell.lower()]
            misspellable_words.remove(word_to_misspell)

        return word_list

    def random_letter_case(word, num_case_changes):
        """Randomly changes the case of letters in a word."""
        for _ in range(num_case_changes):
            if len(word) > 0:
                case_change_type = random.choice(['single', 'all'])
                if case_change_type == 'single':
                    pos = random.randint(0, len(word) - 1)
                    word = word[:pos] + (word[pos].upper() if word[pos].islower() else word[pos].lower()) + word[pos+1:]
                else:  # Change all
                    word = word.swapcase()
        return word

    # Load a dictionary of common misspelled words (this should be pre-loaded or loaded from a file)
    cmw_dict = {
        'definitely': 'definately',
        'separate': 'seperate',
        'occurrence': 'occurence',
        'government': 'goverment',
        'receive': 'recieve'
    }

    # Split the query into words for easier manipulation
    words = query.split()

    # Apply CMW modifications first
    if 'CMW' in typo_dict:
        words = apply_cmw(words, typo_dict['CMW'], cmw_dict)

    modified_query = ' '.join(words)

    # Iterate over each word and apply character-level modifications
    for modification, num in typo_dict.items():
        if modification == 'insertion':
            for _ in range(num):
                word_idx = random.randint(0, len(words) - 1)
                words[word_idx] = random_insertion(words[word_idx], 1)
        elif modification == 'deletion':
            for _ in range(num):
                word_idx = random.randint(0, len(words) - 1)
                words[word_idx] = random_deletion(words[word_idx], 1)
        elif modification == 'replacement':
            for _ in range(num):
                word_idx = random.randint(0, len(words) - 1)
                words[word_idx] = random_replacement(words[word_idx], 1)
        elif modification == 'repetition':
            for _ in range(num):
                word_idx = random.randint(0, len(words) - 1)
                words[word_idx] = random_repetition(words[word_idx], 1)
        elif modification == 'swapping':
            for _ in range(num):
                word_idx = random.randint(0, len(words) - 1)
                words[word_idx] = random_swapping(words[word_idx], 1)
        elif modification == 'LCC':
            for _ in range(num):
                word_idx = random.randint(0, len(words) - 1)
                words[word_idx] = random_letter_case(words[word_idx], 1)

    # Reconstruct the modified query
    modified_query = ' '.join(words)
    
    return modified_query

# Example usage:
query = "Where is the capital of Germany?"
typo_dict = {
    "insertion": 0,
    "deletion": 1,
    "replacement": 0,
    "repetition": 0,
    "swapping": 0,
    "CMW": 0,
    "LCC": 0
}

for _ in range(10):
    modified_query = apply_typo_modifications(query, typo_dict)
    print(modified_query)

Where is the capital of Gerany?
Whre is the capital of Germany?
Where is the caital of Germany?
Were is the capital of Germany?
Where is the capitl of Germany?
Where is the capital of German?
Where is te capital of Germany?
Where is the capital of Germany?
Where is the cpital of Germany?
Where is the capital of Gemany?


## 3. Semantic Similarity

In [3]:
from gensim.models import KeyedVectors
from nlpia.data.loaders import get_data, BIGDATA_PATH

wordvector_path = os.path.join(BIGDATA_PATH, 'GoogleNews-vectors-negative300.bin.gz')

# Load a pre-trained word2vec model (this is just an example, the path should be to your downloaded model)
word_vectors = KeyedVectors.load_word2vec_format(wordvector_path, binary=True)

# Function to find similar words
def find_similar_words(word, topn=10):
    try:
        similar_words = word_vectors.most_similar(positive=[word], topn=topn)
        return [word for word, similarity in similar_words]
    except KeyError:
        return []

# Example usage
related_words = find_similar_words('philosopher')
print(related_words)

ImportError: cannot import name 'Mapping' from 'collections' (/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/collections/__init__.py)

In [3]:
from nltk.corpus import wordnet as wn
import nltk

# Download WordNet data
nltk.download('wordnet')

def find_related_nouns(word):
    related_nouns = set()
    for synset in wn.synsets(word, pos=wn.NOUN):
        # Traverse through the hyponyms (subordinate concepts) and hypernyms (superordinate concepts)
        for lemma in synset.lemmas():
            related_nouns.add(lemma.name())
        for hypernym in synset.hypernyms():
            for lemma in hypernym.lemmas():
                related_nouns.add(lemma.name())
    return related_nouns

# Example usage
related_words = find_related_nouns('philosopher')
print(related_words)

[nltk_data] Downloading package wordnet to
[nltk_data]     /nfs/homedirs/daro/nltk_data...


{'student', 'scholar', 'scholarly_person', 'soul', 'individual', 'bookman', 'philosopher', 'person', 'somebody', 'mortal', 'someone'}


In [4]:
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to find semantically similar words
def find_similar_phrases(word, context_word):
    # Encode word and context into embeddings
    word_embedding = model.encode(word)
    context_embedding = model.encode(context_word)

    # Compute cosine similarity
    similarity_score = util.pytorch_cos_sim(word_embedding, context_embedding).item()
    
    return similarity_score

# Example usage
similarity = find_similar_phrases('philosopher', 'philosophy')
print(similarity)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


0.8044580817222595
